# 33 — Feature Engineering for Regression (Objective 3)

**Objective.** Test a few simple business-derived features for predicting shipment weight, using training rows only.

**Input:** `data/processed/regression_model_input.csv` and `feature_engine/model_features.csv`.

**Output:** `feature_engine/candidate_features.csv` and `training_and_evaluation/feature_engineering_probe.csv`.

This notebook evaluates five candidate derived features on training data. None is carried forward into the model, keeping the predictor set simple and easy to explain.

## 0. Setup

In [ ]:
import os
for var in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(var, "1")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mplconfig")

import matplotlib
matplotlib.use("Agg")

import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()
p = obj_paths(3)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

## 1. Create candidate features

Feature engineering can improve predictive accuracy by creating new variables that capture business relationships not visible in the raw columns. For this shipment-weight regression, simple combinations such as a composite operating-problem count or an infrastructure score may explain variance in the target that individual columns miss. All candidate features are assessed on training rows only, so any decision about adding or excluding them carries no risk of look-ahead bias.

In [ ]:
df = pd.read_csv(p["processed"] / "regression_model_input.csv")
features = pd.read_csv(p["feature_engine"] / "model_features.csv")
base_features = features["feature"].tolist()
train = df["split"].eq("train")
max_year = df["wh_est_year"].max()

candidates = df.copy()
candidates["warehouse_age"] = max_year - candidates["wh_est_year"]
candidates["operating_problem_total"] = candidates["storage_issue_reported_l3m"] + candidates["wh_breakdown_l3m"] + candidates["transport_issue_l1y"]
candidates["shops_per_distributor"] = candidates["retail_shop_num"] / candidates["distributor_num"]
candidates["staff_per_distributor"] = candidates["workers_num"] / candidates["distributor_num"]
candidates["infrastructure_score"] = candidates["electric_supply"] + candidates["temp_reg_mach"] + candidates["flood_proof"] - candidates["flood_impacted"]

candidate_sources = {
    "warehouse_age": ["wh_est_year"],
    "operating_problem_total": ["storage_issue_reported_l3m", "wh_breakdown_l3m", "transport_issue_l1y"],
    "shops_per_distributor": ["retail_shop_num", "distributor_num"],
    "staff_per_distributor": ["workers_num", "distributor_num"],
    "infrastructure_score": ["electric_supply", "temp_reg_mach", "flood_proof", "flood_impacted"],
}

copy_rows = []
for candidate, source_cols in candidate_sources.items():
    copy_model = LinearRegression().fit(candidates.loc[train, source_cols], candidates.loc[train, candidate])
    copy_rows.append({
        "candidate": candidate,
        "business_idea": {
            "warehouse_age": "age is easier to read than establishment year",
            "operating_problem_total": "single count of reported operating problems",
            "shops_per_distributor": "market load per distributor",
            "staff_per_distributor": "staff intensity relative to distribution network",
            "infrastructure_score": "simple site-readiness score",
        }[candidate],
        "source_columns": ", ".join(source_cols),
        "r2_from_source_columns": copy_model.score(candidates.loc[train, source_cols], candidates.loc[train, candidate]),
    })
copy_table = pd.DataFrame(copy_rows)
save_table(copy_table, p["feature_engine"] / "candidate_features.csv", index=False)
display(copy_table.round(4))

> **Interpretation.**
>
> - Three candidates are exact re-expressions of existing columns.
> - The two ratio features are mostly explained by their source columns.
> - A derived feature must improve prediction enough to justify making the model less direct.

## 2. Probe whether candidates improve prediction

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"r2": "r2", "mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"}

probe_rows = []
sets_to_test = [("base", base_features)] + [(candidate, base_features + [candidate]) for candidate in candidate_sources]
for candidate_set, feature_list in sets_to_test:
    X = candidates.loc[train, feature_list]
    y = candidates.loc[train, "product_wg_ton"]
    ridge_pipeline = Pipeline([
        ("prep", ColumnTransformer([("scale", StandardScaler(), feature_list)], verbose_feature_names_out=False)),
        ("model", Ridge(alpha=1.0)),
    ])
    for model_name, estimator in [
        ("Ridge", ridge_pipeline),
        ("Random forest", RandomForestRegressor(n_estimators=160, random_state=RANDOM_STATE, n_jobs=1, min_samples_leaf=2, max_features="sqrt")),
    ]:
        result = cross_validate(estimator, X, y, cv=cv, scoring=scoring, n_jobs=1)
        probe_rows.append({
            "candidate_set": candidate_set,
            "model": model_name,
            "r2_mean": result["test_r2"].mean(),
            "mae_mean": -result["test_mae"].mean(),
            "rmse_mean": -result["test_rmse"].mean(),
        })
probe_table = pd.DataFrame(probe_rows)
base = probe_table[probe_table["candidate_set"].eq("base")].set_index("model")
probe_table["delta_r2_vs_base"] = probe_table.apply(lambda row: row["r2_mean"] - base.loc[row["model"], "r2_mean"], axis=1)
probe_table["delta_mae_vs_base"] = probe_table.apply(lambda row: row["mae_mean"] - base.loc[row["model"], "mae_mean"], axis=1)
probe_table = probe_table.sort_values(["model", "delta_r2_vs_base"], ascending=[True, False])
save_table(probe_table, p["train_eval"] / "feature_engineering_probe.csv", index=False)
display(probe_table.round(4))

> **Interpretation.**
>
> - Ridge cross-validation R² is essentially flat across all candidate sets, so no derived feature improves the linear baseline.
> - The random forest gains most from `operating_problem_total`, but that feature is a perfect sum of existing problem columns and adds no new information.
> - The small, inconsistent deltas confirm that none of the five candidates is worth carrying forward.

> **Decision — candidate features.**
>
> - `warehouse_age`, `operating_problem_total` and `infrastructure_score` are exact re-expressions of existing columns.
> - `shops_per_distributor` and `staff_per_distributor` add no meaningful Ridge improvement.
> - `operating_problem_total` helps the random forest mechanically, but it is a perfect sum of existing problem columns and adds no new business information.
> - Keeping the original 29 predictors is simpler and easier to defend.

## 3. Save outputs

In [ ]:
# Both output files were written inside the analysis cells above.
print(f"candidate_features.csv         → {p['feature_engine'] / 'candidate_features.csv'}")
print(f"feature_engineering_probe.csv  → {p['train_eval'] / 'feature_engineering_probe.csv'}")

## 4. Checks

In [ ]:
_candidates = pd.read_csv(p["feature_engine"] / "candidate_features.csv")
assert _candidates.shape == (5, 4), f"unexpected shape {_candidates.shape}"
assert _candidates.isnull().sum().sum() == 0, "nulls in candidate_features"

_probe = pd.read_csv(p["train_eval"] / "feature_engineering_probe.csv")
assert _probe.shape == (12, 7), f"unexpected shape {_probe.shape}"
assert _probe.isnull().sum().sum() == 0, "nulls in feature_engineering_probe"

print("checks passed")

---
## Summary

- Five candidate features were tested on training rows only.
- None is added to the regression model input.
- The model comparison notebook tests standard regression families against the 29 encoded predictors from the earlier data preparation step.